# FundusNet — Colab Training Pipeline

Train 5 models (Swin, EfficientNet V2, ConvNeXt V2, DeiT III, MaxViT) on GPU,
export to ONNX, verify, and save to Google Drive.

**Runtime:** GPU (T4) required. Go to `Runtime > Change runtime type > T4 GPU`.

**Dataset:** Upload `retina_dataset.zip` to Google Drive before running.

**Time:** ~7-8 hours for all 5 models (80 epochs, 3-fold CV each).

## 1. Setup — Clone repo & install dependencies

In [ ]:
!git clone https://github.com/Mariakevin/FundusNet.git
%cd FundusNet

!pip install -q -r requirements.txt
!pip install -q timm onnxruntime

print("Setup complete!")

## 2. Mount Google Drive & extract dataset

In [ ]:
import os

from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

ZIP_PATH = "/content/drive/MyDrive/retina_dataset.zip"

if not os.path.exists(ZIP_PATH):
    print(f"ERROR: {ZIP_PATH} not found.")
    print("Upload retina_dataset.zip to Google Drive first!")
else:
    size_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f"Found {ZIP_PATH} ({size_mb:.0f} MB)")

    # Check if already extracted
    if os.path.exists("/content/retina_dataset") and len(os.listdir("/content/retina_dataset")) > 1:
        print("Dataset already extracted — skipping.")
    else:
        print("Extracting...")
        !unzip -q "{ZIP_PATH}" -d /content/
        print("Done!")

# Verify dataset
DATASET = "/content/retina_dataset"
print(f"\nDataset: {DATASET}")
total = 0
for cls in ["Healthy", "Cataract", "Glaucoma", "Retina Disease"]:
    cls_path = os.path.join(DATASET, cls)
    if os.path.exists(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tiff"))])
        print(f"  {cls}: {count} images")
        total += count
    else:
        print(f"  {cls}: MISSING!")
print(f"  Total: {total} images")

## 3. Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ERROR: No GPU detected!")
    print("Go to Runtime > Change runtime type > T4 GPU")

## 4. Train All 5 Models (Sequential with Progress)

Trains one model at a time. Each model:
1. Trains (80 epochs, 3-fold CV)
2. Exports to ONNX with verification
3. Saves to Google Drive immediately

**Estimated time:** ~7-8 hours total.

**Tip:** Keep this tab open and laptop plugged in to avoid disconnection.

In [ ]:
import os
import subprocess
import sys
import threading
import time

# ============================================================
# KEEP-ALIVE: Prints dot every 60s to prevent idle disconnect
# ============================================================
keep_alive_running = True


def keep_alive():
    while keep_alive_running:
        time.sleep(60)
        sys.stdout.write(".")
        sys.stdout.flush()


alive_thread = threading.Thread(target=keep_alive, daemon=True)
alive_thread.start()
print("Keep-alive started.")

# ============================================================
# TRAINING CONFIG
# ============================================================
MODELS = ["swin", "efficientnet_v2", "convnext_v2", "deit3", "maxvit"]
DATASET = "/content/retina_dataset"
DRIVE_DIR = "/content/drive/MyDrive"
EPOCHS = 80
FOLDS = 3
BATCH_SIZE = 32

print("=" * 60)
print("FundusNet - All Models Training Pipeline")
print("=" * 60)
print(f"Models:     {len(MODELS)}")
print(f"Epochs:     {EPOCHS}")
print(f"Folds:      {FOLDS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Dataset:    {DATASET}")
print("Est time:   ~7-8 hours")
print("=" * 60)

start_time = time.time()
completed = []
failed = []

# ============================================================
# TRAIN EACH MODEL
# ============================================================
for i, model in enumerate(MODELS, 1):
    elapsed = time.time() - start_time
    remaining = len(MODELS) - i + 1
    pct = (i - 1) / len(MODELS) * 100

    print(f"\n{'#' * 60}")
    print(f"# [{i}/{len(MODELS)}] {pct:.0f}% Complete | Training: {model}")
    print(f"# Elapsed: {elapsed / 3600:.1f} hrs | Est remaining: {remaining * 1.5:.1f} hrs")
    print(f"{'#' * 60}\n")

    # Skip if already exported to Drive
    onnx_drive = f"{DRIVE_DIR}/{model}_retinopathy.onnx"
    if os.path.exists(onnx_drive):
        print("  Already on Drive — skipping.")
        completed.append(model)
        continue

    model_start = time.time()

    # --- TRAIN ---
    cmd = f"python train.py --models {model} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --folds {FOLDS} --dataset {DATASET}"
    print(f"Running: {cmd}\n")
    result = subprocess.run(cmd, shell=True)

    checkpoint = f"experiments/models/{model}_best.pth"

    if not os.path.exists(checkpoint):
        print(f"\nWARNING: Checkpoint not found for {model} — training may have failed.")
        failed.append(model)
        continue

    # --- EXPORT TO ONNX ---
    onnx_local = f"models/{model}_retinopathy.onnx"
    print(f"\nExporting {model} to ONNX...")
    export_cmd = f"python export_onnx.py --model {model} --checkpoint {checkpoint} --output {onnx_local} --verify"
    subprocess.run(export_cmd, shell=True)

    # --- SAVE TO DRIVE ---
    if os.path.exists(onnx_local):
        os.system(f"cp {onnx_local} {DRIVE_DIR}/")
        model_time = time.time() - model_start
        print(f"\n{'=' * 60}")
        print(f"DONE: {model} in {model_time / 60:.1f} min")
        print(f"Saved: {onnx_drive}")
        print(f"{'=' * 60}")
        completed.append(model)
    else:
        print(f"\nWARNING: ONNX export failed for {model}.")
        failed.append(model)

# ============================================================
# SUMMARY
# ============================================================
keep_alive_running = False
total_time = time.time() - start_time

print(f"\n{'=' * 60}")
print("TRAINING COMPLETE")
print(f"{'=' * 60}")
print(f"Total time: {total_time / 3600:.1f} hours")
print(f"Completed:  {len(completed)}/{len(MODELS)}")
if completed:
    print(f"  Models: {', '.join(completed)}")
if failed:
    print(f"Failed:    {', '.join(failed)}")
print(f"{'=' * 60}")

# List ONNX files on Drive
print("\nModels saved to Google Drive:")
!ls -lh /content/drive/MyDrive/*_retinopathy.onnx 2>/dev/null || echo "  None found"

## 5. Verify Exported ONNX Models

In [ ]:
import os

print("Models on Google Drive:")
print("-" * 50)
for f in sorted(os.listdir("/content/drive/MyDrive")):
    if f.endswith(".onnx"):
        size_mb = os.path.getsize(f"/content/drive/MyDrive/{f}") / 1e6
        print(f"  {f}: {size_mb:.1f} MB")

print("\nLocal models/ folder:")
print("-" * 50)
if os.path.exists("models"):
    for f in sorted(os.listdir("models")):
        if f.endswith(".onnx"):
            size_mb = os.path.getsize(f"models/{f}") / 1e6
            print(f"  {f}: {size_mb:.1f} MB")
else:
    print("  No local models found.")

## 6. Quick Inference Test (ONNX Runtime)

In [ ]:
import time

import numpy as np
import onnxruntime as ort

categories = ["Healthy", "Cataract", "Glaucoma", "Retina Disease"]

print("Inference test:")
print("-" * 60)

for model in ["swin", "efficientnet_v2", "convnext_v2", "deit3", "maxvit"]:
    path = f"models/{model}_retinopathy.onnx"
    if not os.path.exists(path):
        print(f"  {model}: NOT FOUND")
        continue

    sess = ort.InferenceSession(path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)

    # Warmup
    for _ in range(3):
        sess.run(None, {"input": dummy})

    # Benchmark
    start = time.time()
    for _ in range(10):
        out = sess.run(None, {"input": dummy})
    ms = (time.time() - start) / 10 * 1000

    probs = np.exp(out[0][0]) / np.exp(out[0][0]).sum()
    pred = categories[np.argmax(probs)]
    conf = max(probs) * 100
    print(f"  {model}: {pred} ({conf:.1f}%) - {ms:.1f}ms")

print("-" * 60)

## 7. Download ONNX Files to Your Computer

Run this cell to download all 5 ONNX models to your local machine.
Save them to `C:\Users\April\Desktop\retina_project\models\`.

In [ ]:
from google.colab import files

print("Downloading ONNX models...")
for f in sorted(os.listdir("/content/drive/MyDrive")):
    if f.endswith(".onnx"):
        print(f"  Downloading {f}...")
        files.download(f"/content/drive/MyDrive/{f}")

print("\nDone! Save files to C:\\Users\\April\\Desktop\\retina_project\\models\\")

## 8. Resume Training (if disconnected)

If Colab disconnected during training, re-run cells 1-4 above.
The training loop automatically skips models that are already on Drive.

In [ ]:
print("Models status:")
print("-" * 50)
for model in ["swin", "efficientnet_v2", "convnext_v2", "deit3", "maxvit"]:
    onnx_drive = f"/content/drive/MyDrive/{model}_retinopathy.onnx"
    checkpoint = f"experiments/models/{model}_best.pth"
    if os.path.exists(onnx_drive):
        print(f"  {model}: COMPLETE (ONNX on Drive)")
    elif os.path.exists(checkpoint):
        print(f"  {model}: PARTIAL (checkpoint exists, needs export)")
    else:
        print(f"  {model}: NOT STARTED")